*An analysis of the top ten. Every agent up there plays a route that varies from its own past runs, in one specific channel, at repeatable positions. Here is how to find those positions in someone else's replay without playing a single game.*

## Where we had got to

Three things about this competition are already established, and each of them came from treating the field as a living population rather than as a set of independent optimisers.

**There are species.** [The leaderboard behaves like a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient): what you meet at one rating band is not what you meet at another, and the bands are populated by recognisable types rather than by a continuum of skill.

**Most of the field shares an opening.** [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) shows hundreds of unrelated teams emitting the same actions, turn for turn, deep into the episode. It is common descent, not convergence.

**And the top diversifies.** Those shared openings do not survive to the end. Agreement is near total in the first days and falls apart in the last ones, which is where the strong agents separate themselves from everybody playing the same script.

**So the question this notebook asks is the obvious next one: how does it diversify?** Not when, which is known, and not by how much, which is known. In what, exactly. Which parts of a route move when a lineage explores, and which parts never move at all.

## The observation that started it, which we first took for a bug

Watching top replays back to back, something did not fit. Two episodes of the same submission, against two different opponents, produced **different action stream hashes** and yet finished with **exactly the same farm**: the same crops on the same squares, the same structures, the same land unlocked.

That should not happen. Our instrument for deciding whether an agent is fixed or adaptive compares action streams, so those two episodes were classified as adaptive, and an adaptive agent is one that reacts, and an agent that reacts to two different opponents has no business building an identical farm both times. The first assumption was that the board comparison was broken.

It was not broken. Both facts were true at once, and reconciling them is the whole notebook: **the stream varies and the plan does not, because the variation lives somewhere that does not touch a tile.** What looked like a defect in the instrument was the instrument measuring the wrong object, and the right object turned out to be far more useful than the one we were after.

## What follows from it

A recorded route is not one indivisible object, which is how it is usually treated: you replay it or you do not. Part of it is a construction plan the agent repeats episode after episode, and part of it is a channel the agent varies freely while the plan holds. Line up enough episodes from one lineage and the two separate cleanly, position by position. This notebook measures that split turn by turn, for the ten highest rated submissions on the ladder at the moment of the snapshot, over 1,482 of their real episodes.

**The term this notebook proposes for the second part is the route's genetic potential**: the positions at which the route's own recorded history licenses a change. It is a property of the recording, measurable before any change is attempted, and section 9 is emphatic that it is not the same thing as the recording being good.

**The result in one line.** All ten of the top agents mutate themselves, and the amount differs by a factor of seven. Two of them carry close to 20 percent of the episode as genetic potential. For the other eight it is between 2.6 and 4.9 percent, and there is nothing in between.

**And a distinction that only appeared once it was measured.** Three of the ten hold their construction plan almost perfectly rigid. Two of those three also run a market that varies constantly; the third barely varies at all. Keeping one plan and running a reactive seller turn out to be **two independent properties**, not one, and section 6 is about the agent that separated them. Section 6 also has the agent that fits none of it, which happens to be the one winning.

**Why none of this is obvious.** The naive way to compare two episodes is to line them up turn by turn and count disagreements. Section 2 shows why that test is wrong for this question, and section 7 measures what it would have got wrong.

---

## 1. The question, stated so it can be answered

An agent playing this game emits three things every turn: what the farmer does, what each of the ten hands does, and what to buy or sell. The first two build a farm. The third moves goods.

Suppose you hold a recording of a strong agent and you want to change it. Any turn you touch falls into one of three classes, and they are the same three a geneticist would name:

* **silent**, because the engine ignores the change and the episode is unaffected;
* **tolerated**, because the agent itself has done something different at that same point in another episode and still finished with the same farm;
* **lethal**, because that turn is load bearing and moving it breaks everything downstream of it.

Only the middle class is usable, and the interesting claim is that it can be identified from recordings alone, with no games played. If an agent varied a turn itself across its own episodes while its plan held, then that position tolerates substitution. That is evidence, not a guess, and it is read off the same kind of data a conservation analysis is read off: many sequences from one lineage, lined up.

**So the object being measured is a map**: for each turn of a chosen base episode, is this position fixed, is it observed to vary, or is it a position where the plan itself moved. The second class is what the title calls the route's **genetic potential**, and the reason for that name is in section 10.

One framing note, because it is easy to read this as being about one agent. The field these ten agents belong to has been described elsewhere as a population rather than a set of individuals: [the leaderboard behaves like a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient), and [most of the field plays a shared opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) before scattering. What follows is the same population seen from inside a single lineage: not how the field differs from itself, but how much one agent differs from itself, and where.

## 2. Turn by turn is the wrong comparison, and this is the correction the whole notebook rests on

Two episodes of one agent both run exactly 720 turns, so it is tempting to compare index `t` against index `t` and count where they differ.

That test fails on the case it most needs to handle. Suppose an agent waits one extra turn early on because a weed appeared where it wanted to walk, and then executes the identical plan one turn later than usual. The plan is the same plan. Index by index the two episodes now disagree almost everywhere, and the measurement reports a total rewrite.

The fix is an **order preserving alignment**: find the longest common blocks between the two sequences and allow those blocks to sit at different offsets. It is the same machinery a text diff uses and the same idea a sequence alignment uses, and it answers a strictly better question, which is not "did turn 300 match turn 300" but "does this action appear, in this order, in both".

That choice also makes a real hypothesis testable rather than assumed. Blocks of construction may be delayed, interleaved or absent, but they should never be **translocated**, because construction funds itself and doing it out of order breaks the economy that pays for the next step. A rearrangement would show up here as alignment failing while the same actions are present on both sides in a different order. Section 7 measures exactly that, and finds it essentially absent.

In [ ]:
import json, sys, warnings
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

# Team names here are Japanese, Cyrillic and Latin. Matplotlib's default font
# covers Cyrillic and draws empty boxes for CJK, which is how the first version
# of this notebook published its most interesting agent with no readable name.
# Use a CJK capable font if the image has one, and label by rank and submission
# when it does not, rather than printing tofu.
from matplotlib import font_manager

def _cjk_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    for n in ("Noto Sans CJK JP", "Noto Sans JP", "IPAGothic", "TakaoGothic",
              "Hiragino Sans", "Arial Unicode MS", "Noto Sans CJK SC"):
        if n in have:
            return n
    return None

CJK = _cjk_font()
if CJK:
    plt.rcParams["font.sans-serif"] = [CJK] + list(
        plt.rcParams.get("font.sans-serif", []))
print("CJK capable font:", CJK or "none, falling back to rank and submission id")

def flabel(team, rank, sub):
    """A label that will actually render in a figure."""
    if CJK or all(ord(c) < 0x2E80 for c in str(team)):
        return str(team)
    return f"team {sub}"

# Find the data by looking for it, never by hardcoding a path. A previous
# notebook in this series died at cell one because an input was mounted under a
# different directory than the one written into the code.
def find_dir(marker, roots=("/kaggle/input", ".", "..", "data")):
    hits = []
    for r in roots:
        p = Path(r)
        if p.exists():
            hits += [q.parent for q in p.rglob(marker)]
    if not hits:
        raise FileNotFoundError(
            f"could not find {marker!r} under {roots}. Attach the dataset.")
    return sorted(hits, key=lambda q: len(str(q)))[0]

BASE = find_dir("streams.parquet")
print("reading from", BASE)

episodes = pd.read_csv(BASE / "episodes.csv",
                       dtype={"episode_id": str, "submission": str})
streams = pd.read_parquet(BASE / "streams.parquet")
streams["episode_id"] = streams["episode_id"].astype(str)
streams["submission"] = streams["submission"].astype(str)

print(f"{len(episodes):,} episodes, {len(streams):,} turns, "
      f"{episodes.submission.nunique()} submissions")

### The field being measured

One row per episode of the ten submissions that held the top ten places on the ladder at the moment of the snapshot. Every episode is a real ladder game against a real opponent, not a simulation.

Two details that cost real evidence to get right, and both are silent failures rather than errors:

**The seat is not recovered from the team name.** One of these ten teams renamed itself part way through the competition, so matching the leaderboard name against the replay drops every episode it played under its old name. Here the seat comes from the set of names that covers all of a submission's episodes, which is immune to renaming.

**Episodes where a team met itself are excluded.** Each of the ten has exactly one, because both of a team's two most recent submissions receive games and the ladder can pair them together. Such an episode has no opponent seat, so it is dropped rather than assigned one arbitrarily.

In [ ]:
per_sub = (episodes.groupby(["submission", "rank_at_snapshot", "team", "ladder_score"])
           .agg(episodes=("episode_id", "size"),
                opponents=("opponent", "nunique"),
                seat0=("seat", lambda s: int((s == 0).sum())),
                median_bank=("bank", "median"))
           .reset_index().sort_values("rank_at_snapshot"))
per_sub["seat1"] = per_sub.episodes - per_sub.seat0
display(per_sub[["rank_at_snapshot", "team", "ladder_score", "submission",
                 "episodes", "opponents", "seat0", "seat1", "median_bank"]]
        .rename(columns={"rank_at_snapshot": "rank"})
        .style.hide(axis="index").format({"median_bank": "{:,.0f}"}))

## 3. Deducing a baseline, since nobody publishes one

Comparing episodes against a baseline needs a baseline, and there is none. The agent's source is not available and its intended route is not written down anywhere. What is available is many sequences from one lineage, which is the situation a **reference sequence** is normally derived from.

So it is deduced, by the least inventive method available: take the **medoid**, the recorded episode whose alignment score against all the others is highest. Two properties make this the right choice over a consensus built position by position:

* it is a **real episode**, so it is by construction a legal sequence of actions. A position by position consensus of two legal routes need not be legal, and a synthetic baseline would inject states the engine rejects. This is the same reason a reference genome is assembled from real reads rather than from a majority vote over incompatible haplotypes.
* its score is **self diagnosing**. An agent that runs one plan has a medoid near 1. An agent that has no single plan cannot have a good medoid, and the number says so without anyone deciding a threshold.

That second property matters more than it sounds, because it means the method reports its own inapplicability instead of returning a confident answer on an agent the idea does not fit.

All episodes are restricted to a single seat first. The two farms are not the same object, so pooling seats measures the geometry of the board rather than the policy of the agent.

In [ ]:
PER_SUB = 90     # episodes considered per submission, before the seat split

streams["plan"] = streams["farmer"] + "|" + streams["hands_h"]
# Key on the PAIR: an episode played between two of these ten submissions
# appears twice, once per seat, and grouping on the episode id alone silently
# concatenates them into one 1,440 turn stream.
by_ep = {k: g.sort_values("turn")
         for k, g in streams.groupby(["submission", "episode_id"])}

def pool_for(sub):
    e = episodes[episodes.submission == sub].sort_values("episode_id").head(PER_SUB)
    seat = int(e.seat.value_counts().idxmax())
    e = e[e.seat == seat]
    out = []
    for _, r in e.iterrows():
        g = by_ep[(sub, r.episode_id)]
        out.append({"episode": r.episode_id, "seat": seat, "opponent": r.opponent,
                    "bank": r.bank, "plan": list(g.plan), "market": list(g.market)})
    return out

def medoid(pool):
    score = []
    for i, a in enumerate(pool):
        s = sum(SequenceMatcher(a=a["plan"], b=b["plan"], autojunk=False).ratio()
                for j, b in enumerate(pool) if i != j)
        score.append((s / (len(pool) - 1), i))
    score.sort(reverse=True)
    return pool[score[0][1]], score[0][0], score[0][0] - score[1][0]

pools = {s: pool_for(s) for s in per_sub.submission}
bases = {s: medoid(p) for s, p in pools.items()}

tab = []
for _, r in per_sub.iterrows():
    b, sc, mg = bases[r.submission]
    tab.append({"rank": r.rank_at_snapshot, "team": r.team, "seat": b["seat"],
                "episodes used": len(pools[r.submission]),
                "baseline episode": b["episode"],
                "medoid score": round(sc, 4), "margin over runner up": round(mg, 5)})
display(pd.DataFrame(tab).style.hide(axis="index"))

The medoid score already separates the field before a single mutation has been considered. Three agents sit at 0.98 and above, meaning any episode of theirs is an almost perfect stand in for any other. One sits at 0.47, meaning it has no single plan for a baseline to approximate, and every statement made below about that agent should be read as a statement about a method being applied outside its range.

## 4. Three states, and the conservation map they make

Align every episode of a submission against its baseline, and label each turn of the baseline. What comes out is a **conservation map**, in the ordinary sense: positions that never vary across the lineage are the constrained ones, and positions that vary are the ones that tolerate substitution.

| | meaning |
|---|---|
| **same** | the plan action aligned and the market order matched too |
| **market mutation** | the plan action aligned here and the market order **differed**. The source varied this turn itself, with its plan intact |
| **plan differs** | the plan action did not align at all. Off limits for this family of change |

The middle state is the whole point. It is not a place where a mutation might work; it is a place where the source has already made one and finished with the same farm. Those are the variable sites of this lineage, and the pale stretches between them are the conserved ones.

*On the colours.* The three are separated by lightness as well as hue, in a monotone order, so the map survives colour vision deficiency, greyscale printing and a bad projector. A legend is always present, and the composition is repeated as a bar chart in section 5 so that identity never rests on colour alone.

*On the labels.* Team names here are Japanese, Cyrillic and Latin. The plotting font on this image has no Japanese glyphs and would draw empty boxes, so where that happens the figures fall back to `team <submission id>` and the tables above carry the real name against the same id.

In [ ]:
SAME, MUT, PLAN = "#DCEEE1", "#B45309", "#0F172A"
CMAP = ListedColormap([SAME, MUT, PLAN])
STATE = {"G": 0, "Y": 1, "R": 2}

def align(a, b):
    m = {}
    for i, j, n in SequenceMatcher(a=a, b=b, autojunk=False).get_matching_blocks():
        for k in range(n):
            m[i + k] = j + k
    return m

def classify(sub):
    base, _, _ = bases[sub]
    rows, shifts, moved, unaligned = [], [], 0, 0
    for o in pools[sub]:
        if o is base:
            continue
        m = align(base["plan"], o["plan"])
        st = []
        for i in range(len(base["plan"])):
            j = m.get(i)
            if j is None:
                st.append("R")
            else:
                shifts.append(j - i)
                st.append("G" if base["market"][i] == o["market"][j] else "Y")
        rows.append("".join(st))
        used = set(m.values())
        left = Counter(base["plan"][i] for i in range(len(base["plan"])) if i not in m)
        right = Counter(o["plan"][j] for j in range(len(o["plan"])) if j not in used)
        moved += sum((left & right).values())
        unaligned += sum(left.values())
    n = len(base["plan"])
    per_turn = [Counter(r[i] for r in rows) for i in range(n)]
    empty = (json.dumps(None), json.dumps([]))
    free = [i for i, c in enumerate(per_turn) if c["R"] == 0 and c["Y"] > 0]
    frozen = [i for i, c in enumerate(per_turn) if c["R"] == 0 and c["Y"] == 0]
    broken = [i for i, c in enumerate(per_turn) if c["R"] > 0]
    traded = [i for i in frozen if base["market"][i] not in empty]
    return {"base": base, "rows": rows, "per_turn": per_turn, "turns": n,
            "free": free, "frozen": frozen, "broken": broken,
            "frozen_traded": traded, "shifts": shifts,
            "moved": moved, "unaligned": unaligned}

surf = {s: classify(s) for s in per_sub.submission}
print("classified", sum(len(v["rows"]) for v in surf.values()),
      "episode comparisons against 10 baselines")

In [ ]:
order = per_sub.sort_values("rank_at_snapshot")
name = {sub: flabel(t, r, sub) for sub, t, r
        in zip(order.submission, order.team, order.rank_at_snapshot)}
rank = dict(zip(order.submission, order.rank_at_snapshot))

show = [order.submission.iloc[1], order.submission.iloc[7],
        order.submission.iloc[8], order.submission.iloc[0]]

fig, axes = plt.subplots(len(show), 1, figsize=(13.5, 9.2),
                         gridspec_kw={"hspace": 0.55})
for ax, sub in zip(axes, show):
    v = surf[sub]
    M = np.array([[STATE[c] for c in r] for r in v["rows"]], dtype=np.int8)
    ax.imshow(M, aspect="auto", cmap=CMAP, vmin=0, vmax=2, interpolation="nearest")
    pct = 100 * len(v["free"]) / v["turns"]
    ax.set_title(f"#{rank[sub]}  {name[sub]}      "
                 f"{len(v['rows'])} episodes against baseline {v['base']['episode']}"
                 f"      mutable on {len(v['free'])} turns ({pct:.0f}% of the episode)",
                 loc="left", fontsize=10.5, pad=6)
    ax.set_ylabel("episode", fontsize=9)
    ax.set_xlim(0, v["turns"])
    for d in range(0, 721, 120):
        ax.axvline(d, color="white", lw=0.6, alpha=0.35)
    ax.tick_params(labelsize=8)
axes[-1].set_xlabel("turn  (720 turns = 30 in game days, gridlines every 5 days)",
                    fontsize=9)
fig.legend(handles=[Patch(facecolor=SAME, label="same as baseline"),
                    Patch(facecolor=MUT, label="market mutation observed"),
                    Patch(facecolor=PLAN, label="plan differs, off limits")],
           loc="upper center", bbox_to_anchor=(0.5, 1.005), ncol=3, frameon=False,
           fontsize=10)
fig.suptitle("Every episode of an agent, aligned against one of its own",
             y=1.055, fontsize=13.5)
plt.show()

Read the rows as episodes and the columns as turns. Two shapes appear and they are not variations of one thing.

The first two panels are almost entirely pale, with vertical amber stripes: the plan is the same plan in every episode, and the agent varies its trading at specific, repeated moments. Those stripes are the mutable surface, and they are **columns** rather than scattered dots, which means the positions are a property of the route and not of the particular game.

The third panel is pale almost everywhere and has hardly any amber at all. Same rigidity of plan, almost no variation in trading, which is the distinction section 6 is about.

**The last panel is the one to look at.** It goes dark around turn 180 and never really comes back. There is no baseline that the other episodes resemble, so most of the episode is off limits, and the weight of that darkness is the whole diagnosis: the panel is not showing an agent that mutates a lot, it is showing an agent for which the concept does not apply. That is not a fixed plan with a seller on top, and no amount of alignment will make it one.

Notice also **where** it goes dark. The pale stretch runs to roughly turn 180, which is a little past the point where the opponent first becomes observable, and the darkness begins there and deepens to the end. An agent whose behaviour is indistinguishable from its own past while it cannot see anyone, and different from it once it can, is the shape a reactive policy makes.

## 5. The genetic potential of each route

Turn the map into one number per agent: of the 720 turns, how many are positions where the source itself has been seen to trade differently with its plan intact. That count is the route's genetic potential, in turns.

The count is also worth normalising by the turns on which the baseline trades at all. A turn where the agent never places an order is not a mutation site in the same sense, since changing it means inventing an action rather than altering one.

In [ ]:
rows = []
for _, r in order.iterrows():
    v = surf[r.submission]
    n = v["turns"]
    trading = len(v["free"]) + len(v["frozen_traded"])
    rows.append({
        "rank": r.rank_at_snapshot, "team": r.team,
        "medoid": round(bases[r.submission][1], 3),
        "plan aligned everywhere": len(v["free"]) + len(v["frozen"]),
        "mutable turns": len(v["free"]),
        "mutable % of episode": round(100 * len(v["free"]) / n, 1),
        "turns the baseline trades": trading,
        "mutable % of trading turns": round(100 * len(v["free"]) / trading, 1) if trading else 0.0,
        "off limits": len(v["broken"]),
    })
gen = pd.DataFrame(rows)
display(gen.sort_values("mutable turns", ascending=False)
        .style.hide(axis="index")
        .background_gradient(subset=["mutable % of episode"], cmap="Oranges"))

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.0))
o = gen.sort_values("mutable turns")
y = np.arange(len(o))
free = o["mutable turns"].values
froz = o["plan aligned everywhere"].values - free
off = o["off limits"].values

ax.barh(y, free, color=MUT, label="market mutation observed", height=0.66)
ax.barh(y, froz, left=free, color=SAME, label="plan held, market never varied",
        height=0.66, edgecolor="white", linewidth=0.8)
ax.barh(y, off, left=free + froz, color=PLAN, label="plan differs, off limits",
        height=0.66, edgecolor="white", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels([f"#{r}  {t[:26]}" for r, t in zip(o["rank"], o["team"])],
                   fontsize=9)
for i, f in enumerate(free):
    ax.text(f + 8, i, f"{f}", va="center", fontsize=9, color="#0F172A")
ax.set_xlabel("turns of the 720 turn episode", fontsize=9.5)
ax.set_xlim(0, 760)
ax.set_title("How much of each route is free for a market only change",
             loc="left", fontsize=12.5, pad=10)
ax.legend(loc="lower right", frameon=False, fontsize=9.5)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(left=False, labelsize=9)
plt.show()

## 6. Two axes, not one, and the agent that shows they are two

The temptation at this point is to declare one family and be done. Three quantities are available, each computed from a different thing:

* the **medoid score**, from aligning action streams;
* the **mutable share**, from comparing market orders at aligned positions;
* the **worst board gap**, from comparing final farms tile by tile, which never looks at an action at all.

Two of them agree on a partition. The third does not, and the disagreement is the finding rather than a defect.

In [ ]:
boards = pd.read_csv(BASE / "boards.csv", dtype={"episode_id": str, "submission": str})

def worst_board_gap(sub):
    """Largest tile disagreement between any two final boards, within a seat.

    Distance rather than identity, because "two distinct boards" is a blunt
    test: a hundred tiles differing in one is the same plan with a weed in the
    way, and a hundred differing in forty is a different farm.
    """
    b = boards[boards.submission == sub].merge(
        episodes[["submission", "episode_id", "seat"]],
        on=["submission", "episode_id"], suffixes=("", "_e"))
    worst = 0
    for seat, g in b.groupby("seat"):
        keys = [str(x).split("/")[0].replace("|", ",").split(",") for x in g.board]
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                if len(keys[i]) == len(keys[j]):
                    worst = max(worst, sum(1 for a, c in zip(keys[i], keys[j]) if a != c))
    return worst

gen["worst board gap (tiles of 100)"] = [worst_board_gap(s) for s in order.submission]

# the plan axis: fixed if the farms it builds barely move between episodes
gen["plan fixed"] = gen["worst board gap (tiles of 100)"] <= 15
gen["market reactive"] = gen["mutable turns"] > 100

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
panels = [("medoid", "medoid score of the baseline", (0.4, 1.02)),
          ("worst board gap (tiles of 100)", "worst final board gap, tiles", None),
          ("mutable % of episode", "mutable share of the episode, %", None)]
for k, (ax, (col, lab, xlim)) in enumerate(zip(axes, panels)):
    v = gen.sort_values(col)
    colours = [MUT if a else (SAME if b else PLAN)
               for a, b in zip(v["market reactive"], v["plan fixed"])]
    ax.scatter(v[col], np.linspace(-0.2, 0.2, len(v)), c=colours, s=115,
               edgecolor="#334155", linewidth=0.7, zorder=3)
    for x, yy, r in zip(v[col], np.linspace(-0.2, 0.2, len(v)), v["rank"]):
        ax.annotate(f"#{r}", (x, yy), textcoords="offset points", xytext=(0, 10),
                    ha="center", fontsize=8.5, color="#334155")
    ax.set_xlabel(lab, fontsize=9.5)
    ax.set_yticks([])
    ax.set_ylim(-0.55, 0.55)
    if xlim:
        ax.set_xlim(*xlim)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(labelsize=9)
axes[0].set_ylabel("the ten agents", fontsize=9)
fig.legend(handles=[Patch(facecolor=MUT, label="plan fixed, market reactive"),
                    Patch(facecolor=SAME, label="plan fixed, market near constant"),
                    Patch(facecolor=PLAN, label="plan moves")],
           loc="upper center", bbox_to_anchor=(0.5, 1.10), ncol=3, frameon=False,
           fontsize=9.5)
fig.suptitle("Two of these three agree on a partition. The third splits it again.",
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

fixed = gen[gen["plan fixed"]].sort_values("rank")
moves = gen[~gen["plan fixed"]].sort_values("rank")

# The two axes crossed, printed rather than asserted, because which of the four
# cells is the empty one is the whole claim of this section.
cells = pd.crosstab(gen["plan fixed"], gen["market reactive"])
print("the two axes crossed, agents per cell")
print(cells.rename_axis(index="plan fixed", columns="market reactive"), "\n")

for label, grp in (("plan FIXED", fixed), ("plan MOVES", moves)):
    print(f"{label} ({len(grp)}): ranks {list(grp['rank'])}")
    print(f"   medoid         {sorted(grp['medoid'])}")
    print(f"   board gap      {sorted(grp['worst board gap (tiles of 100)'])}")
    print(f"   mutable turns  {sorted(grp['mutable turns'])}\n")
odd = fixed[~fixed["market reactive"]]
for _, r in odd.iterrows():
    print(f"the exception: #{r['rank']} {r['team']} holds its plan to "
          f"{r['worst board gap (tiles of 100)']} tiles and a medoid of {r['medoid']}, "
          f"yet varies its market on only {r['mutable turns']} turns, "
          f"{r['mutable % of trading turns']} % of the turns it trades on.")

In [ ]:
top = gen[gen["rank"] == 1].iloc[0]
rest = gen[gen["rank"] != 1]
scores = per_sub.set_index("rank_at_snapshot")["ladder_score"].astype(float)
distinct = (boards.groupby(["submission", "seat"]).board_h.nunique()
            .rename("distinct boards").reset_index()
            .merge(per_sub[["submission", "rank_at_snapshot", "team"]], on="submission"))

print(f"rank 1 is {scores.loc[1] - scores.loc[2]:,.1f} rating points clear of rank 2, "
      f"which is more than ranks 2 to 10 span between them "
      f"({scores.loc[2] - scores.loc[10]:,.1f}).\n")
for col in ("medoid", "plan aligned everywhere", "mutable % of episode",
            "worst board gap (tiles of 100)"):
    print(f"  {col:<32} rank 1 = {top[col]:>8}   "
          f"rest of the field {rest[col].min()} to {rest[col].max()}")
print()
d1 = distinct[distinct.rank_at_snapshot == 1]
print("distinct final boards, rank 1 by seat:",
      ", ".join(f"seat {int(r.seat)}: {int(r['distinct boards'])}"
                for _, r in d1.iterrows()))
print("the same figure for everyone else ranges "
      f"{distinct[distinct.rank_at_snapshot != 1]['distinct boards'].min()} to "
      f"{distinct[distinct.rank_at_snapshot != 1]['distinct boards'].max()}")

It builds a different farm nearly every time it plays, it has no episode that stands in for its others, and it is the best agent on the board by a margin larger than the one separating second place from tenth. Whatever it is, it is not the same kind of thing as the other nine, and the honest word for it is that it looks like a different species.

That reading is not proved here and this notebook is the wrong place to try. It would need the field, not ten agents, and a way of relating lineages to each other rather than measuring inside one. The closing section says what those measurements would be.

**That is one observation of one agent, and it should be read as exactly that.** But it is the observation that stops this notebook from having a tidy ending, so it is worth stating without softening: on this sample, the quantity being measured **does not predict strength**. It predicts how much of a route you could safely copy, which is a different thing, and the agent least worth measuring this way is the one winning.

It also means the top of this ladder contains at least one thing that none of the machinery here describes. Everything in sections 3 to 5 assumes there is a plan to find. For that agent there is not, and the medoid score says so at 0.474 before any conclusion is drawn from it. A method that reports its own inapplicability is the most this notebook can offer about rank 1; understanding it needs a different instrument.

**The medoid score and the board gap agree**, and they were computed from different things: one from action streams, the other from farms with the actions never consulted. Three agents hold their plan, seven move it, and the two measures put the same three on the same side. That is worth having on its own, because the medoid costs no games and no engine and predicts the tile by tile comparison.

**And those three are the answer to the observation at the top of this notebook.** Different hash, identical farm, against different opponents, was never a defect: it is what a plan fixed agent looks like through an instrument that only reads action streams. Six to eight tiles of a hundred is the entire distance between any two farms an agent in that group builds, while its stream differs in hundreds of turns.

**The mutable share splits those three again.** Two of them run a market that varies constantly. The third holds its plan just as rigidly and trades almost the same way every time.

So the property this notebook set out to measure is not one thing. Keeping a fixed construction plan and running a reactive seller on top are **independent axes**, and three of the four cells are occupied here: **two** agents hold a fixed plan and run a reactive market, **one** holds a fixed plan with a near constant market, and the remaining **seven** move their plan while varying their market on fewer than a hundred aligned turns.

The empty cell is the other one, **a moving plan with a reactive market**, and it is empty partly by construction rather than as a fact about agents: once the plan moves, most of the episode goes off limits and there are few aligned turns left on which a varying market could be seen at all. Read the count as what this instrument can resolve, not as a claim that such an agent cannot exist.

That distinction matters for the practical question. A fixed plan with a near constant market is close to a pure recording: there is very little of it that its own history licenses you to change. A fixed plan with a reactive market hands you a fifth of the episode.

### And then there is the one at the top

Rank 1 is not a member of either fixed plan cell. On every quantity computed here it sits outside the range of the other nine, and by a margin that does not overlap.

## 7. Do the blocks move? The reordering test

The alignment allows matched blocks to sit at different offsets, so two things can now be asked that the naive comparison could not.

**Are blocks displaced?** Count the offset of every matched position. An offset of zero means the plan sits at the same index in both episodes.

**Are blocks reordered?** When alignment fails, look at the actions left unmatched on each side. If the same actions appear on both sides in a different order, the plan was rearranged. If the leftovers have nothing in common, the agent genuinely did something else.

The second is the interesting one, because a route whose construction can be permuted is a far larger space of legal children than one whose construction is fixed in order.

In [ ]:
rows = []
for _, r in order.iterrows():
    v = surf[r.submission]
    sh = Counter(v["shifts"])
    tot = sum(sh.values())
    rows.append({
        "rank": r.rank_at_snapshot, "team": r.team,
        "matched positions": tot,
        "at zero offset": sh.get(0, 0),
        "% at zero offset": round(100 * sh.get(0, 0) / tot, 2) if tot else 0.0,
        "largest offset seen": max([abs(k) for k in sh if k != 0], default=0),
        "unaligned plan actions": v["unaligned"],
        "explainable as a move": v["moved"],
        "% explainable as a move": round(100 * v["moved"] / v["unaligned"], 2)
                                   if v["unaligned"] else 0.0,
    })
sh = pd.DataFrame(rows)
display(sh.style.hide(axis="index"))

print(f"across all ten agents, {sh['at zero offset'].sum():,} of "
      f"{sh['matched positions'].sum():,} matched positions "
      f"({100*sh['at zero offset'].sum()/sh['matched positions'].sum():.2f} %) "
      f"sit at the same index")
print(f"and {sh['explainable as a move'].sum():,} of "
      f"{sh['unaligned plan actions'].sum():,} unaligned plan actions "
      f"({100*sh['explainable as a move'].sum()/sh['unaligned plan actions'].sum():.2f} %) "
      f"could be explained by a block having moved")

**Construction is not reordered.** At most a little over one percent of the unaligned plan actions of any agent could be accounted for by a block having moved, and for four of the ten the figure is exactly zero. When these agents diverge they do something different, not the same thing later.

**And displacement is nearly absent too**, which is a stronger result than was expected. Under one percent of matched positions sit at a non zero offset, and for four agents the figure is zero. The plans are effectively locked to the index, not merely locked in order. When an offset does occur it can be large, up to 144 turns, so displacement is rare rather than small.

That has a practical consequence worth stating plainly. The expensive alignment and the naive turn by turn comparison return nearly the same answer **for these agents**, but that is a measured fact about them rather than a property of the problem, and it was not knowable before measuring. Applied to an agent that does pause and resume, the naive comparison would report a rewrite where there is none.

## 8. Where the mutable turns are, and what the game says about it

The mutable positions are not spread uniformly. Two structural facts about this game bear on where they should fall.

An agent cannot see its opponent until part way into the episode, so any variation before that point is a response to its own farm: a weed that appeared, a harvest that came in light, a shed that filled at a different rate. It is not, and cannot be, a response to the other player.

And the value of a good in this market falls as you sell into it, so trading is concentrated where the goods exist rather than spread evenly.

In [ ]:
VISIBLE_AT = 136   # the turn at which the opponent first becomes observable

fig, ax = plt.subplots(figsize=(13.5, 4.6))
for k, (_, r) in enumerate(order.iterrows()):
    v = surf[r.submission]
    marks = np.array(v["free"])
    y = len(order) - k
    ax.scatter(marks, np.full(len(marks), y), s=9, marker="|",
               color=MUT if len(v["free"]) > 100 else "#94A3B8")
ax.axvline(VISIBLE_AT, color=PLAN, lw=1.4, ls="--")
ax.text(VISIBLE_AT + 6, len(order) + 0.45,
        "opponent first observable", fontsize=9, color=PLAN)
ax.set_yticks(range(1, len(order) + 1))
ax.set_yticklabels([f"#{r}  {name[s][:24]}" for r, s in
                    zip(order.rank_at_snapshot[::-1], order.submission[::-1])],
                   fontsize=9)
ax.set_xlim(0, 720)
ax.set_xlabel("turn", fontsize=9.5)
ax.set_title("Where the mutable turns sit in the episode",
             loc="left", fontsize=12.5, pad=10)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.tick_params(left=False, labelsize=9)
plt.show()

for _, r in order.iterrows():
    v = surf[r.submission]
    f = np.array(v["free"])
    if len(f) == 0:
        continue
    print(f"#{r.rank_at_snapshot:>2} {r.team[:26]:<28} "
          f"{len(f):>3} mutable turns | "
          f"{int((f < VISIBLE_AT).sum()):>3} before the opponent is visible | "
          f"first at turn {int(f.min()):>3}, last at turn {int(f.max()):>3}")

## 9. Tolerated is not beneficial, and this is where the analogy earns its keep

The map says where the source varied. It does not say that any particular replacement is an improvement, and it would be a serious misreading to treat a variable site as an invitation.

The biology says the same thing and says it first. A site being **tolerant of substitution is not evidence that any substitution improves the organism.** Most changes at a tolerated site are neutral, a good share are mildly deleterious, and the beneficial ones are rare enough that finding them is the entire difficulty. Conservation tells you where selection is not acting; it is silent about which direction to move.

And there is direct evidence here that crude substitutions are not free. Three rules for rewriting a route's market orders were built and measured against the same recorded field, each behind its own flag, with the control verified bit identical first. **All three lost.** The most aggressive of them, emptying the shed whenever the route mentioned an item, lost about 150,000 a game across 80 paired games. The mechanism is arithmetic rather than accident: revenue in this market falls as you sell into it, so dumping at the first opportunity is close to the worst schedule available.

So the honest statement of what this notebook produces is narrow, and worth keeping narrow:

* **it does establish** that a large, identifiable and repeated part of certain routes is a channel the source itself varies, that the part is findable from recordings at zero cost in games, and that construction is never rearranged;
* **it does not establish** that the recorded values at those sites are replaceable by anything better. A conservation map tells you where the molecule tolerates a change. It does not tell you which change to make.

The next measurement is the one that separates those, and it is not a variation of this one: generate the market orders from scratch, from the agent's own shed and money, keeping construction verbatim, and compare. If the result holds up, the plan is the load bearing half and the schedule is replaceable. If it collapses, the recorded schedule is doing work that a simple rule cannot.

## 10. How far the analogy goes, and exactly where it stops

The vocabulary above has been doing real work rather than decorating, so it is worth saying which parts transfer and which do not, because the most famous part is the part that does not.

**What transfers.** Targeted genome editing locates its site **by sequence match**, not by coordinate, which is precisely the role alignment plays here and the reason section 2 rejects index comparison. It also cannot cut anywhere: a specific adjacent motif has to be present for the machinery to bind at all. The equivalent constraint here is measured rather than assumed, and it is that the market channel can only be touched at positions where the plan aligned.

**What does not transfer, and it is the cut and paste part.** Moving material is exactly the operation section 7 finds absent, at under half a percent across every agent measured, and the standing hypothesis is that a rearrangement would break the economy that funds construction. The streams are also of fixed length, one action per turn for 720 turns, so insertions and deletions do not exist in this representation at all. **Every legal change here is a substitution at a fixed locus**, which puts the operation closer to base editing than to a double strand break.

So the honest framing is the older and less glamorous one. Line up many sequences from one lineage; the positions that never vary are under functional constraint, the variable positions tolerate substitution, and the map of which is which is how you decide where a molecule can be altered without destroying it. Sections 4 and 5 compute that map, one route at a time, and section 9 is the caveat that has always come with it.

**Why the framework keeps paying.** This is the third time that treating this competition as a biological system rather than an optimisation problem has produced a measurement the optimisation framing does not suggest. [The ladder as a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient) found a population where a ranking was expected. [A shared opening across the field](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) found common descent where independent design was expected. This one finds conserved and variable sites inside a lineage where an indivisible recording was expected. The vocabulary is not a metaphor laid over the results afterwards; it is what suggested looking.

---

## 11. The questions this leaves open, and they are all phylogeny

Everything above is **within** one lineage: how much an agent differs from its own past. It says nothing about how the lineages are related **to each other**, and once the vocabulary is biological the next questions name themselves. They are left open on purpose; each of them is a measurement rather than a speculation, and none of them fits here.

**What is the most recent common ancestor of two routes?** For sequences this is unusually tractable, because it does not have to be inferred from present-day variation the way an MRCA normally does. Two agents agree turn for turn and then stop: the turn where they stop **is** the split. Over a field that gives a matrix of split times directly, and from it a tree.

**Is there an Eve, and is there a LUCA?** The first is the ancestral opening from which every agent currently at the top descends, in the sense that mitochondrial Eve is the most recent common ancestor along one line of inheritance. The second is the root of the whole field, and there is a specific candidate for it: the first published notebook whose route the field adopted. Both are datable from the episode record, which makes the fossil record here complete in a way no real one is.

**Does the field even have a tree?** Almost certainly not, and for exactly the reason bacterial phylogeny is hard. Teams read each other's public notebooks and take code from them, which is **horizontal transfer**: a lineage acquires a trait from a distant relative rather than from its parent. A history with enough of it is a network and not a tree, and the interesting quantity is not the tree but **how far from a tree it is**.

**How fast does a lineage drift, and does it ratchet?** The mutable positions found here are the sites where a lineage is free to move. Whether the field's occupants actually move at those sites over weeks, and whether the changes accumulate in one direction or wander, is a question the episode record can answer and this snapshot cannot.

**And where does rank 1 sit in all of it?** Section 6 says it looks like a different species and declines to prove it. The phylogenetic version of that question is sharper and answerable: does it split from the dominant lineage late, which would make it a descendant that left, or does it share no ancestor with it at all, which would make it something that never belonged. Those are different histories and they leave different traces.

**A caution that belongs with all of them.** Sequence similarity is evidence of shared ancestry only where independent invention is unlikely, and here it often is not: [the opening is close to forced](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening), so agreement in the first days is weak evidence of anything. Any tree built from this data has to be built from where lineages **stop** agreeing, not from how much they agree.

---

## Where this sits, and what to read next

This is one lineage seen from inside. Three earlier notebooks look at the same field from outside, and the picture only makes sense with both.

* [**Everyone is playing the same opening**](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) measures how much of the field shares a schedule, and when they stop sharing it. That is the between agent version of the alignment done here within one agent, and it is why an early match proves much less than a late one.
* [**The leaderboard is a habitat gradient**](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient) treats the ladder as a population rather than a ranking. The two cells found in section 6 are types within that population, and whether they are stable types or a passing fashion is a question that file is better equipped to ask.
* [**What kind of game is this?**](https://www.kaggle.com/code/destbreso/kaggriculture-what-kind-of-game-is-this) derives the turn at which the opponent first becomes observable, which is the line drawn in section 8, and the reason variation before it cannot be a reaction to the other player.
* [**The free experiment you already ran**](https://www.kaggle.com/code/destbreso/kaggriculture-the-free-experiment-you-already-ran) is the method note for treating already played episodes as a designed experiment, which is what this notebook does with 1,482 of them.
* [**Wins, not money**](https://www.kaggle.com/code/destbreso/wins-not-money) is why none of the banks above are used to rank anything.

**Data.** A fixed snapshot of every public episode played by the ten highest rated submissions at the time it was taken, reduced to per turn actions and final boards. It is a snapshot on purpose: the ladder moves daily, and rebuilding it would change every number above.